In [0]:
bronze_df = spark.read.table("workspace.default.bronze_yellow_taxi")

print(f"Bronze table count: {bronze_df.count()}")
display(bronze_df.limit(5))

Bronze table count: 3724889


VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,ingestion_timestamp,file_name
2,2026-01-01T00:54:04.000,2026-01-01T00:59:37.000,1,0.97,1,N,239,238,1,7.2,1.0,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.0,2026-10-11T15:32:50.367Z,dbfs:/Volumes/workspace/default/landing_zone/yellow_tripdata_2026-01.parquet
1,2026-01-01T00:34:04.000,2026-01-01T00:39:47.000,0,0.9,1,N,163,162,2,7.9,4.25,0.5,0.0,0.0,1.0,13.65,2.5,0.0,0.75,2026-10-11T15:32:50.367Z,dbfs:/Volumes/workspace/default/landing_zone/yellow_tripdata_2026-01.parquet
1,2026-01-01T00:57:06.000,2026-01-01T01:05:59.000,0,1.4,1,N,43,237,1,10.7,4.25,0.5,2.5,0.0,1.0,18.95,2.5,0.0,0.75,2026-10-11T15:32:50.367Z,dbfs:/Volumes/workspace/default/landing_zone/yellow_tripdata_2026-01.parquet
2,2026-01-01T00:15:22.000,2026-01-01T00:58:10.000,4,5.58,1,N,142,209,1,38.7,1.0,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75,2026-10-11T15:32:50.367Z,dbfs:/Volumes/workspace/default/landing_zone/yellow_tripdata_2026-01.parquet
2,2026-01-01T00:27:13.000,2026-01-01T00:40:43.000,0,2.16,1,N,88,144,1,13.5,1.0,0.5,3.85,0.0,1.0,23.1,2.5,0.0,0.75,2026-10-11T15:32:50.367Z,dbfs:/Volumes/workspace/default/landing_zone/yellow_tripdata_2026-01.parquet


In [0]:
from pyspark.sql.functions import col, round, timestamp_diff, lit

# Calculate trip duration directly in minutes
duration_in_minutes = timestamp_diff(lit("MINUTE"), col("tpep_pickup_datetime"), col("tpep_dropoff_datetime"))

silver_df = bronze_df.withColumn("trip_duration_minutes", round(duration_in_minutes, 2))

display(silver_df.select(
    "tpep_pickup_datetime", 
    "tpep_dropoff_datetime", 
    "trip_duration_minutes", 
    "trip_distance"
).limit(5))

tpep_pickup_datetime,tpep_dropoff_datetime,trip_duration_minutes,trip_distance
2026-01-01T00:54:04.000,2026-01-01T00:59:37.000,5,0.97
2026-01-01T00:34:04.000,2026-01-01T00:39:47.000,5,0.9
2026-01-01T00:57:06.000,2026-01-01T01:05:59.000,8,1.4
2026-01-01T00:15:22.000,2026-01-01T00:58:10.000,42,5.58
2026-01-01T00:27:13.000,2026-01-01T00:40:43.000,13,2.16


### **DATA QUALITY AND BOUNDARY FILTERS**

In [0]:
#FILTER OUT DIRTY & IMPOSSIBLE RECORDS
silver_filtered_df = silver_df.filter(
    (col("trip_distance")>0) & (col("trip_distance")<200) & (col("passenger_count")>0)
    & (col("fare_amount")>2.50) & (col("fare_amount")<1000) & (col("trip_duration_minutes")>= 1) & (col("trip_duration_minutes")<= 360)
)
print(f"Rows before filter:{silver_df.count()}")
print(f"Rows after filter:{silver_filtered_df.count()}")


Rows before filter:3724889
Rows after filter:2497229


### **DEDUPLICATION**

In [0]:
# 1. Deduplicate records based on trip identity columns
silver_deduped_df = silver_filtered_df.dropDuplicates([
    "VendorID", "tpep_pickup_datetime", "PULocationID", "DOLocationID"
])

# 2. Read Bronze taxi zones reference table
zones_df = spark.read.table("workspace.default.bronze_taxi_zones")

# 3. Enrich pickup and dropoff zone details
silver_enriched_df = silver_deduped_df \
    .join(zones_df.select(
        col("LocationID").alias("PULocationID"),
        col("Borough").alias("pickup_borough"),
        col("Zone").alias("pickup_zone")
    ), on="PULocationID", how="left") \
    .join(zones_df.select(
        col("LocationID").alias("DOLocationID"),
        col("Borough").alias("dropoff_borough"),
        col("Zone").alias("dropoff_zone")
    ), on="DOLocationID", how="left")

display(silver_enriched_df.select(
    "tpep_pickup_datetime", "pickup_borough", "pickup_zone",
    "dropoff_borough", "dropoff_zone", "trip_distance", "fare_amount"
).limit(20))

tpep_pickup_datetime,pickup_borough,pickup_zone,dropoff_borough,dropoff_zone,trip_distance,fare_amount
2026-01-01T00:52:50.000,Manhattan,Upper West Side South,Manhattan,Yorkville West,1.17,10.0
2026-01-01T00:31:14.000,Manhattan,Upper West Side North,Manhattan,Bloomingdale,1.39,9.3
2026-01-01T00:11:01.000,Manhattan,East Chelsea,Manhattan,Lincoln Square East,2.96,32.4
2026-01-01T00:31:35.000,Manhattan,Penn Station/Madison Sq West,Manhattan,Lenox Hill West,2.29,19.1
2026-01-01T00:18:10.000,Manhattan,East Chelsea,Manhattan,Greenwich Village South,1.73,11.4
2026-01-01T00:28:51.000,Manhattan,Lower East Side,Manhattan,East Village,1.02,8.6
2026-01-01T00:21:56.000,Manhattan,Central Park,Manhattan,Midtown North,0.95,8.6
2026-01-01T00:41:51.000,Manhattan,West Chelsea/Hudson Yards,Manhattan,West Village,1.0,11.4
2026-01-01T00:32:23.000,Manhattan,Upper West Side North,Manhattan,Manhattan Valley,1.15,12.8
2026-01-01T00:07:27.000,Manhattan,Yorkville East,Manhattan,Lenox Hill East,1.03,7.9


In [0]:
# -------------------------------------------------------------------------
# Feature Engineering & Business Normalization:
# 1. Temporal attributes: Extract date, hour, and day name for time-series analytics.
# 2. Payment categorization: Decode numeric payment IDs into business descriptions.
# 3. Tip percentage: Calculate tip rate relative to base fare for card transactions.
# -------------------------------------------------------------------------

from pyspark.sql.functions import col, to_date, hour, date_format, when, round

silver_transformed_df = silver_enriched_df \
    .withColumn("pickup_date", to_date(col("tpep_pickup_datetime"))) \
    .withColumn("pickup_hour", hour(col("tpep_pickup_datetime"))) \
    .withColumn("pickup_day_name", date_format(col("tpep_pickup_datetime"), "EEEE")) \
    .withColumn("payment_type_desc", 
        when(col("payment_type") == 1, "Credit Card")
        .when(col("payment_type") == 2, "Cash")
        .when(col("payment_type") == 3, "No Charge")
        .when(col("payment_type") == 4, "Dispute")
        .otherwise("Other")
    ) \
    .withColumn("tip_percentage", 
        when((col("payment_type") == 1) & (col("fare_amount") > 0), 
             round((col("tip_amount") / col("fare_amount")) * 100, 2))
        .otherwise(0.0)
    )

display(silver_transformed_df.select(
    "pickup_date", "pickup_hour", "pickup_day_name", 
    "payment_type_desc", "fare_amount", "tip_amount", "tip_percentage"
).limit(5))



pickup_date,pickup_hour,pickup_day_name,payment_type_desc,fare_amount,tip_amount,tip_percentage
2026-01-01,0,Thursday,Credit Card,10.0,3.0,30.0
2026-01-01,0,Thursday,Cash,9.3,0.0,0.0
2026-01-01,0,Thursday,Credit Card,32.4,7.63,23.55
2026-01-01,0,Thursday,Credit Card,19.1,4.97,26.02
2026-01-01,0,Thursday,Credit Card,11.4,3.43,30.09


In [0]:
from pyspark.sql.functions import current_timestamp

# 1. Add audit metadata timestamp
silver_final_df = silver_transformed_df.withColumn(
    "silver_processed_timestamp", 
    current_timestamp()
)

# 2. Write to Unity Catalog as managed Delta table
silver_table_name = "workspace.default.silver_yellow_taxi"

silver_final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(silver_table_name)

print(f"Table {silver_table_name} saved successfully!")

Table workspace.default.silver_yellow_taxi saved successfully!


In [0]:
# Verify the Silver Delta table from Unity Catalog
silver_table = spark.read.table("workspace.default.silver_yellow_taxi")

print(f"Final Silver Row Count: {silver_table.count()}")
display(silver_table.limit(5))

Final Silver Row Count: 2495712


DOLocationID,PULocationID,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,ingestion_timestamp,file_name,trip_duration_minutes,pickup_borough,pickup_zone,dropoff_borough,dropoff_zone,pickup_date,pickup_hour,pickup_day_name,payment_type_desc,tip_percentage,silver_processed_timestamp
263,239,2,2026-01-01T00:52:50.000,2026-01-01T01:01:18.000,1,1.17,1,N,1,10.0,1.0,0.5,3.0,0.0,1.0,18.0,2.5,0.0,0.0,2026-10-11T15:32:50.367Z,dbfs:/Volumes/workspace/default/landing_zone/yellow_tripdata_2026-01.parquet,8,Manhattan,Upper West Side South,Manhattan,Yorkville West,2026-01-01,0,Thursday,Credit Card,30.0,2026-10-11T18:08:03.326Z
24,238,2,2026-01-01T00:31:14.000,2026-01-01T00:37:52.000,3,1.39,1,N,2,9.3,1.0,0.5,0.0,0.0,1.0,14.3,2.5,0.0,0.0,2026-10-11T15:32:50.367Z,dbfs:/Volumes/workspace/default/landing_zone/yellow_tripdata_2026-01.parquet,6,Manhattan,Upper West Side North,Manhattan,Bloomingdale,2026-01-01,0,Thursday,Cash,0.0,2026-10-11T18:08:03.326Z
142,68,2,2026-01-01T00:11:01.000,2026-01-01T00:48:49.000,2,2.96,1,N,1,32.4,1.0,0.5,7.63,0.0,1.0,45.78,2.5,0.0,0.75,2026-10-11T15:32:50.367Z,dbfs:/Volumes/workspace/default/landing_zone/yellow_tripdata_2026-01.parquet,37,Manhattan,East Chelsea,Manhattan,Lincoln Square East,2026-01-01,0,Thursday,Credit Card,23.55,2026-10-11T18:08:03.326Z
141,186,2,2026-01-01T00:31:35.000,2026-01-01T00:52:46.000,1,2.29,1,N,1,19.1,1.0,0.5,4.97,0.0,1.0,29.82,2.5,0.0,0.75,2026-10-11T15:32:50.367Z,dbfs:/Volumes/workspace/default/landing_zone/yellow_tripdata_2026-01.parquet,21,Manhattan,Penn Station/Madison Sq West,Manhattan,Lenox Hill West,2026-01-01,0,Thursday,Credit Card,26.02,2026-10-11T18:08:03.326Z
114,68,2,2026-01-01T00:18:10.000,2026-01-01T00:28:46.000,4,1.73,1,N,1,11.4,1.0,0.5,3.43,0.0,1.0,20.58,2.5,0.0,0.75,2026-10-11T15:32:50.367Z,dbfs:/Volumes/workspace/default/landing_zone/yellow_tripdata_2026-01.parquet,10,Manhattan,East Chelsea,Manhattan,Greenwich Village South,2026-01-01,0,Thursday,Credit Card,30.09,2026-10-11T18:08:03.326Z
